# Age-Stratified Medical Condition Classification

This notebook demonstrates how to:
1. Train SSAST models on specific age groups and conditions
2. Evaluate models across different age ranges
3. Analyze how acoustic signatures of medical conditions vary with age

## Use Cases

- **Cross-age generalization**: Train on elementary school kids (7-12), test on teenagers (13-17)
- **Focused condition analysis**: Study only respiratory conditions (asthma, allergies, chronic cough)
- **Task-specific models**: Train on sustained phonation tasks only for breath support analysis

In [ ]:
from bridge2ai_ssast.easy_api import train_condition_classifier, evaluate_model
import sys
sys.path.insert(0, '.')

print("✓ Imports successful")

## Example 1: Train on Elementary School Ages (7-12)

Train a model to detect respiratory conditions in younger children.

In [ ]:
# Train on ages 7-12, respiratory conditions only
model_elementary, metrics_elementary = train_condition_classifier(
    dataset='pediatric',
    conditions=['asthma', 'allergies', 'chronic_cough'],
    age_range=(7, 12),  # Elementary school ages
    tasks=['long-sounds'],  # Sustained phonation
    epochs=5,  # Quick demo (use 30+ for real training)
    batch_size=16,
    learning_rate=5e-5,
    output_dir='results/demo_elementary_respiratory/',
    validation_split=0.2,
)

print(f"\nBest validation F1: {metrics_elementary['best_val_f1']:.3f}")

## Example 2: Evaluate on Teenagers (13-17)

Test the elementary-trained model on teenagers to see if acoustic features generalize across age groups.

In [ ]:
# Evaluate on teenagers (13-17) using same conditions and tasks
metrics_teens = evaluate_model(
    model_path='results/demo_elementary_respiratory/best_model.pth',
    dataset='pediatric',
    conditions=['asthma', 'allergies', 'chronic_cough'],
    age_range=(13, 17),  # Teenager ages
    tasks=['long-sounds'],
    batch_size=16,
)

print(f"\nCross-age performance:")
print(f"  Elementary (7-12) validation F1: {metrics_elementary['best_val_f1']:.3f}")
print(f"  Teenagers (13-17) test F1: {metrics_teens['macro_f1']:.3f}")
print(f"  Performance drop: {(metrics_elementary['best_val_f1'] - metrics_teens['macro_f1']):.3f}")

## Example 3: Neurological Conditions (All Ages)

Train on ADHD and depression without age filtering.

In [ ]:
# Train on neurological conditions (no age filter)
model_neuro, metrics_neuro = train_condition_classifier(
    dataset='pediatric',
    conditions=['adhd', 'depression'],
    age_range=None,  # All ages (4-17)
    tasks=None,  # All tasks
    epochs=5,
    batch_size=16,
    output_dir='results/demo_neurological_all_ages/',
)

print(f"\nNeurological conditions (all ages): F1 = {metrics_neuro['best_val_f1']:.3f}")

## Example 4: Adult Age Filtering (30-50 years)

Train on middle-aged adults for comparison with pediatric models.

In [ ]:
# Note: This requires adult dataset preprocessing to be complete
# (mel_128bin_50hz.parquet must exist in adult directory)

try:
    model_adult, metrics_adult = train_condition_classifier(
        dataset='adult',
        conditions=['asthma', 'allergies', 'chronic_cough'],
        age_range=(30, 50),  # Middle-aged adults
        tasks=['passage', 'sentence'],  # Reading tasks
        epochs=5,
        batch_size=16,
        output_dir='results/demo_adult_middle_aged/',
    )
    
    print(f"\nAdult model (ages 30-50): F1 = {metrics_adult['best_val_f1']:.3f}")
except FileNotFoundError as e:
    print(f"\nAdult dataset not yet preprocessed: {e}")
    print("Run: python scripts/convert_linear_to_128mel.py --dataset adult")

## Example 5: Command-Line Usage

You can also use the updated training scripts with age filtering:

```bash
# Train on pediatric ages 7-12 only
python scripts/stage2_train_pediatric.py \
  --adult-checkpoint results/stage1_adult/best_model.pth \
  --epochs 30 \
  --age-min 7 \
  --age-max 12 \
  --output-dir results/stage2_elementary/

# Train on adult ages 40-60 only
python scripts/stage1_train_adult.py \
  --epochs 50 \
  --age-min 40 \
  --age-max 60 \
  --output-dir results/stage1_middle_aged/
```

## Example 6: Analyze Age-Specific Performance

Evaluate the same model across multiple age bins to understand age-related performance trends.

In [ ]:
# Train on all pediatric ages
model_all_ages, metrics_all = train_condition_classifier(
    dataset='pediatric',
    conditions=['asthma', 'allergies'],
    age_range=None,  # Train on all ages
    tasks=['long-sounds'],
    epochs=5,
    output_dir='results/demo_all_ages_asthma_allergies/',
)

# Evaluate on different age bins
age_bins = [
    (4, 6, 'Early childhood'),
    (7, 9, 'Elementary (early)'),
    (10, 12, 'Elementary (late)'),
    (13, 15, 'Teenager (early)'),
    (16, 17, 'Teenager (late)'),
]

print("\nAge-stratified performance:")
print("="*60)
for age_min, age_max, label in age_bins:
    try:
        metrics = evaluate_model(
            model_path='results/demo_all_ages_asthma_allergies/best_model.pth',
            dataset='pediatric',
            conditions=['asthma', 'allergies'],
            age_range=(age_min, age_max),
            tasks=['long-sounds'],
        )
        print(f"{label:25s} (ages {age_min}-{age_max}): F1 = {metrics['macro_f1']:.3f} | "
              f"AUROC = {metrics['macro_auroc']:.3f}")
    except Exception as e:
        print(f"{label:25s} (ages {age_min}-{age_max}): No data available")

print("="*60)

## Summary

### Available Filtering Options

| Parameter | Type | Example | Description |
|-----------|------|---------|-------------|
| `dataset` | str | `'pediatric'` or `'adult'` | Which dataset to use |
| `conditions` | list[str] | `['asthma', 'adhd']` | Which conditions to predict (subset of 8 labels) |
| `age_range` | tuple(int, int) | `(7, 12)` | Age filter (min, max) |
| `tasks` | list[str] | `['long-sounds']` | Acoustic tasks to include |

### 8 Available Conditions

1. `asthma` - Asthma diagnosis
2. `allergies` - Allergies (seasonal for adults, general for pediatric)
3. `hearing_loss` - Hearing loss or impairment
4. `adhd` - ADHD/ADD diagnosis
5. `gerd_reflux` - GERD/LPR/acid reflux
6. `speech_therapy` - Speech/language therapy history
7. `chronic_cough` - Chronic cough diagnosis
8. `depression` - Depression diagnosis

### Age Distributions

- **Pediatric**: Ages 4-17 (peak at age 14 with 37 participants)
- **Adult**: Ages 18-90+ (varies by cohort)

### Research Questions Enabled by Age Filtering

1. **Cross-age generalization**: Do acoustic biomarkers learned from children transfer to adults?
2. **Developmental changes**: How do voice-condition relationships change from childhood to adolescence?
3. **Age-matched controls**: Compare performance with age-appropriate normative data
4. **Pubertal effects**: Compare pre-puberty (7-12) vs. post-puberty (15-17) voice features